In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.neural_network import MLPClassifier

print("Libraries loaded!")

Libraries loaded!


In [2]:
train_df = pd.read_csv("../dataset/KDDTrain+.csv")

print("Original Dataset Shape:", train_df.shape)
print("\nClass Distribution:\n")
print(train_df["class"].value_counts())

Original Dataset Shape: (50000, 25)

Class Distribution:

class
normal          20000
neptune         15000
smurf           10000
guess_passwd     5000
Name: count, dtype: int64


In [3]:
y = train_df["class"]

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

with open("../model/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

print("Label encoding done!")
print(pd.Series(y).value_counts())

Label encoding done!
2    20000
1    15000
3    10000
0     5000
Name: count, dtype: int64


In [4]:
X = train_df.drop(["class", "classnum"], axis=1, errors="ignore")

drop_cols = ["source_ip", "destination_ip", "timestamp"]
X = X.drop(columns=[col for col in drop_cols if col in X.columns])

categorical_cols = ["protocol_type", "service", "flag", "attack_category"]
existing_cats = [col for col in categorical_cols if col in X.columns]

X = pd.get_dummies(X, columns=existing_cats)

X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

feature_columns = X.columns.tolist()

with open("../model/feature_columns.pkl", "wb") as f:
    pickle.dump(feature_columns, f)

print("Feature processing done!")
print("Feature shape:", X.shape)

Feature processing done!
Feature shape: (50000, 30)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (40000, 30)
Test shape: (10000, 30)


In [6]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

with open("../model/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Scaling completed!")

Scaling completed!


In [7]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=20,
    min_samples_split=3,
    min_samples_leaf=1,
    class_weight="balanced_subsample",
    random_state=42
)

svm = SGDClassifier(
    loss="log_loss",
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    max_iter=300,
    random_state=42
)

ada = AdaBoostClassifier(
    n_estimators=150,
    random_state=42
)

print("Models initialized!")

Models initialized!


In [8]:
rf.fit(X_train, y_train)
svm.fit(X_train, y_train)
mlp.fit(X_train, y_train)
ada.fit(X_train, y_train)

print("Models trained!")

d:\anaconda\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Models trained!


In [9]:
rf_pred = rf.predict(X_test)
svm_pred = svm.predict(X_test)
mlp_pred = mlp.predict(X_test)
ada_pred = ada.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
svm_acc = accuracy_score(y_test, svm_pred)
mlp_acc = accuracy_score(y_test, mlp_pred)
ada_acc = accuracy_score(y_test, ada_pred)

print("RF Accuracy :", rf_acc)
print("SVM Accuracy:", svm_acc)
print("MLP Accuracy:", mlp_acc)
print("ADA Accuracy:", ada_acc)

RF Accuracy : 0.9957
SVM Accuracy: 0.9928
MLP Accuracy: 0.9936
ADA Accuracy: 0.8


In [10]:
print("\nConfusion Matrix (RF):\n")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report (RF):\n")
print(classification_report(y_test, rf_pred))


Confusion Matrix (RF):

[[1000    0    0    0]
 [   0 2993    0    7]
 [   0    0 4000    0]
 [   0   36    0 1964]]

Classification Report (RF):

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1000
           1       0.99      1.00      0.99      3000
           2       1.00      1.00      1.00      4000
           3       1.00      0.98      0.99      2000

    accuracy                           1.00     10000
   macro avg       1.00      0.99      1.00     10000
weighted avg       1.00      1.00      1.00     10000



In [11]:
rf_prob = rf.predict_proba(X_test)
svm_prob = svm.predict_proba(X_test)
mlp_prob = mlp.predict_proba(X_test)
ada_prob = ada.predict_proba(X_test)

accuracies = {
    "rf": rf_acc,
    "svm": svm_acc,
    "mlp": mlp_acc,
    "ada": ada_acc
}

total = sum(accuracies.values())
weights = {model: acc / total for model, acc in accuracies.items()}

final_prob = (
    weights["rf"] * rf_prob +
    weights["svm"] * svm_prob +
    weights["mlp"] * mlp_prob +
    weights["ada"] * ada_prob
)

y_pred_final = np.argmax(final_prob, axis=1)

print("\nFinal Ensemble Accuracy:",
      accuracy_score(y_test, y_pred_final))


Final Ensemble Accuracy: 0.9954


In [12]:
with open("../model/model_weights.pkl", "wb") as f:
    pickle.dump(weights, f)

pickle.dump(rf, open("../model/rf_model.pkl", "wb"))
pickle.dump(svm, open("../model/svm_model.pkl", "wb"))
pickle.dump(mlp, open("../model/mlp_model.pkl", "wb"))
pickle.dump(ada, open("../model/ada_model.pkl", "wb"))

print("All models saved successfully!")

All models saved successfully!
